<div align="center">

<a href="https://colab.research.google.com/github/utkukose/cyber-physical-systems-NB-lecture/blob/main/exams/final/FIN2_battery_dispatch.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final option 2: A battery with forecast-based dispatch

**Cyber-Physical Systems (BIL-435 Siber Fiziksel Sistemler)**<br>Final project: A networked cyber-physical system with its digital twin. Starter Colab notebook

[Project page](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/exams/final) | [Course repository](https://github.com/utkukose/cyber-physical-systems-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A supermarket with 120 kWp of solar panels and a battery of 200 kWh. Replace the values marked `# TODO` with the data of your own system. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. Forecasting the load and the solar power

*Lecture: [Week 10, Regression: Forecasting the load](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#regression) and [Week 10, Forecasting the generation](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#generation)*

The battery is planned one day ahead, so it needs forecasts of the load and of the solar power for every hour. The
load forecast is a linear regression. It has one coefficient for each hour of an open day and of a Sunday, and a
cooling term that grows with the outdoor temperature above 15 °C:

$$\hat{L}_t = \beta_{h(t),\,s(t)} + \gamma\,\max(0,\ T_t - 15).$$

The solar forecast scales the clear-sky power $P_{cs}$ with the forecast cloud cover $c$ in the model of Kasten and
Czeplak:

$$\hat{P}_{pv} = P_{cs}\,\left(1 - 0.75\,c^{3.4}\right).$$

The regression learns from all weeks but the last one and forecasts the last week with the forecast temperatures. The
mean absolute error (MAE), the root mean square error (RMSE) and the mean absolute percentage error (MAPE) measure the
forecast. The seasonal naive forecast, the load of the same hour one week earlier, is the baseline that a useful
forecast must beat.

In [ ]:
# TODO: replace the simulated history with the hourly log of your site
P_PV_PEAK = 120.0            # rated power of the solar panels (kWp)

def market_history(weeks=8, seed=31):
    """A simulated hourly log of a supermarket that opens 8:00 to 22:00, and 10:00 to 20:00 on Sundays."""
    rng = np.random.default_rng(seed)
    days = weeks * 7
    t = np.arange(days * 24)
    hour, day = t % 24, t // 24
    sunday = day % 7 == 6
    temp = 18 + 6 * np.sin(2 * np.pi * (hour - 9) / 24) + np.repeat(rng.normal(0, 3, days), 24)
    cloud = np.clip(np.repeat(rng.beta(1.2, 1.8, days), 24) + rng.normal(0, 0.1, t.size), 0, 1)
    clear = P_PV_PEAK * 0.8 * np.clip(np.sin(np.pi * (hour - 6) / 14), 0, None)
    shop = np.where(sunday, (hour >= 10) & (hour < 20), (hour >= 8) & (hour < 22))
    load = 45 + 70 * shop + 1.2 * np.clip(temp - 15, 0, None) + rng.normal(0, 4, t.size)
    return pd.DataFrame({"hour": t, "temp": temp, "temp_fc": temp + rng.normal(0, 1.5, t.size), "cloud": cloud,
                         "cloud_fc": np.clip(cloud + rng.normal(0, 0.15, t.size), 0, 1), "clear": clear,
                         "pv": clear * (1 - 0.75 * cloud ** 3.4), "load": load})

H = market_history()
TRAIN, TEST = H.iloc[:-168], H.iloc[-168:]

def features(d, temp_col):
    """One column per hour of an open day and of a Sunday, and the cooling term max(0, T - 15)."""
    hour = d.hour.values % 24
    sunday = (d.hour.values // 24) % 7 == 6
    X = np.zeros((len(d), 49))
    X[np.arange(len(d)), hour + 24 * sunday] = 1.0
    X[:, 48] = np.clip(d[temp_col].values - 15, 0, None)
    return X

def errors(actual, forecast):
    e = np.asarray(forecast) - np.asarray(actual)
    return {"MAE (kW)": round(float(np.mean(np.abs(e))), 2), "RMSE (kW)": round(float(np.sqrt(np.mean(e ** 2))), 2),
            "MAPE (%)": round(float(100 * np.mean(np.abs(e) / np.abs(actual))), 2)}

coef, *_ = np.linalg.lstsq(features(TRAIN, "temp"), TRAIN.load.values, rcond=None)
LOAD_FC = features(TEST, "temp_fc") @ coef
PV_FC = TEST.clear.values * (1 - 0.75 * TEST.cloud_fc.values ** 3.4)
naive = H.load.values[-336:-168]
day = TEST.clear.values > 0
print(f"Cooling term: {coef[48]:.2f} kW per K above 15 °C")
print(pd.DataFrame([{"forecast": "load, regression", **errors(TEST.load, LOAD_FC)},
                    {"forecast": "load, seasonal naive", **errors(TEST.load, naive)}]).to_string(index=False))
pv_mae = float(np.mean(np.abs(PV_FC[day] - TEST.pv.values[day])))
print(f"Solar power, daylight hours: MAE {pv_mae:.2f} kW")
RESULTS["load forecast: MAE (kW)"] = errors(TEST.load, LOAD_FC)["MAE (kW)"]
RESULTS["seasonal naive: MAE (kW)"] = errors(TEST.load, naive)["MAE (kW)"]
RESULTS["solar forecast: MAE (kW)"] = round(pv_mae, 2)

## 2. The dispatch as a linear program

*Lecture: [Week 9, Optimisation: Variables, objective and constraints](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#optimisation) and [Week 9, Battery scheduling](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#battery-schedule)*

Each day, the energy management system plans the battery for the next 24 hours with a linear program. For every hour
$t$, its variables are the import $g_t$, the export $x_t$, the charging power $c_t$, the discharging power $d_t$ and the
energy $e_t$ at the end of the hour. It minimises the cost of the day

$$\min \sum_t \left(\pi_t\,g_t - \pi_f\,x_t\right)$$

with the price $\pi_t$ of the hour and the feed-in payment $\pi_f$. The constraints are the power balance and the energy
of the battery:

$$g_t - x_t = L_t - P_{pv,t} + c_t - d_t, \qquad e_t = e_{t-1} + \eta\,c_t - d_t/\eta .$$

The bounds keep the powers within the limits of the inverter and the energy between the reserve $e_{min}$ and the
capacity. The day must end with at least the energy of its start. The cell plans each day of the last week from the
forecasts and evaluates the plan with the actual load and solar power. It compares the plans with no battery and with a
plan that knew the actual values in advance.

In [ ]:
# TODO: the battery and the tariff of your site
E_CAP = 200.0                # usable energy of the battery (kWh)
P_BAT = 100.0                # largest charging and discharging power (kW)
ETA = 0.95                   # efficiency of charging and of discharging
E_MIN = 60.0                 # reserve for the cold rooms during a grid outage (kWh), see Section 5
E0 = 100.0                   # energy at midnight (kWh)
FEED_IN = 0.05               # payment for energy fed into the grid (euro/kWh)
PRICE = np.array([0.10] * 6 + [0.20] * 11 + [0.30] * 5 + [0.10] * 2)    # euro/kWh for the hours 0 to 23

def schedule(load, pv, e0=E0):
    """The linear program of one day; variables g, x, c, d and e, 24 of each."""
    n = len(load)
    cost = np.r_[PRICE, -FEED_IN * np.ones(n), np.zeros(3 * n)]
    A_eq, b_eq = [], []
    for t in range(n):
        row = np.zeros(5 * n)
        row[[t, n + t, 2 * n + t, 3 * n + t]] = [1, -1, -1, 1]           # g - x - c + d = L - P_pv
        A_eq.append(row)
        b_eq.append(load[t] - pv[t])
        row = np.zeros(5 * n)
        row[4 * n + t] = 1                                               # e_t - e_(t-1) - eta c + d / eta = 0
        if t > 0:
            row[4 * n + t - 1] = -1
        row[[2 * n + t, 3 * n + t]] = [-ETA, 1 / ETA]
        A_eq.append(row)
        b_eq.append(e0 if t == 0 else 0.0)
    bounds = [(0, None)] * (2 * n) + [(0, P_BAT)] * (2 * n) + [(E_MIN, E_CAP)] * (n - 1) + [(e0, E_CAP)]
    res = linprog(cost, A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bounds, method="highs")
    g, x, c, d, e = np.split(res.x, 5)
    return {"c": c, "d": d, "e": e, "planned cost": res.fun}

def day_cost(load, pv, c, d):
    """Cost of a day (euro) with the actual load and solar power and the battery powers c and d."""
    net = load - pv + c - d
    return float(np.sum(np.where(net > 0, PRICE * net, FEED_IN * net)))

rows, PLANS = [], []
for k in range(7):
    a = TEST.iloc[24 * k: 24 * (k + 1)]
    plan = schedule(LOAD_FC[24 * k: 24 * (k + 1)], PV_FC[24 * k: 24 * (k + 1)])
    best = schedule(a.load.values, a.pv.values)
    PLANS.append(plan)
    zero = np.zeros(24)
    rows.append({"day": k + 1, "no battery": day_cost(a.load.values, a.pv.values, zero, zero),
                 "plan from forecasts": day_cost(a.load.values, a.pv.values, plan["c"], plan["d"]),
                 "plan with actual values": day_cost(a.load.values, a.pv.values, best["c"], best["d"]),
                 "cycles": float(plan["d"].sum() / E_CAP)})
week = pd.DataFrame(rows).round(2)
print(week.to_string(index=False))
saving = week["no battery"].sum() - week["plan from forecasts"].sum()
print(f"Saving of the week: {saving:.2f} euro; with perfect forecasts {week['no battery'].sum() - week['plan with actual values'].sum():.2f} euro")
RESULTS["cost of the week without a battery (euro)"] = round(week["no battery"].sum(), 2)
RESULTS["saving of the week, plan from forecasts (euro)"] = round(saving, 2)
RESULTS["saving of the week, perfect forecasts (euro)"] = round(week["no battery"].sum() - week["plan with actual values"].sum(), 2)
RESULTS["equivalent full cycles per day"] = round(week["cycles"].mean(), 3)

p = PLANS[0]
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(TEST.load.values[:24], label="load")
ax[0].plot(TEST.pv.values[:24], label="solar power")
ax[0].bar(range(24), p["c"] - p["d"], color="#9AA5AE", label="battery: charge minus discharge")
ax[1].plot(p["e"], color="#1E6FA8", label="planned energy")
ax[1].axhline(E_MIN, color="#B23B3B", ls="--", lw=1, label="reserve")
ax[0].set_ylabel("Power (kW)")
ax[1].set_ylabel("Energy (kWh)")
ax[1].set_xlabel("Hour of the first day")
for a in ax:
    a.legend(fontsize=8)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Reading the devices over Modbus

*Lecture: [Week 8, Modbus: Polling the field devices](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#modbus)*

The energy management system reads the meter, the inverter and the battery management system over Modbus. The devices
share one RS-485 (Recommended Standard 485) line in the RTU (remote terminal unit) mode. A byte takes 11 bits on the line, so its time is $t_c = 11/B$ at $B$ bits per second. A request to read
registers has 8 bytes, the answer with $N$ registers has $5 + 2N$ bytes, and each frame ends with a silence of 3.5
character times. A device needs the time $t_p$ to answer. One poll therefore takes

$$t_{poll} = (8 + 3.5)\,t_c + t_p + (5 + 2N + 3.5)\,t_c ,$$

and the master polls the devices one after another. The cell computes the cycle for two bit rates. It checks that the
meter is read often enough for the fast control of the battery, which follows the meter within the hour.

In [ ]:
# TODO: the field bus of your site
DEVICES = [("meter", 20), ("inverter", 30), ("battery management", 40)]   # registers read in each poll
T_PROC_MS = 10.0             # time that a device needs to answer (ms)
NEED_MS = 1000.0             # the battery control needs a fresh reading of the meter at least this often (ms)

def poll_ms(registers, baud):
    t_c = 11 / baud * 1000
    return (8 + 3.5) * t_c + T_PROC_MS + (5 + 2 * registers + 3.5) * t_c

rows = []
for baud in [9600, 19200]:
    polls = {name: poll_ms(n, baud) for name, n in DEVICES}
    cycle = sum(polls.values())
    rows.append({"bit rate": baud, **{f"{k} (ms)": round(v, 1) for k, v in polls.items()},
                 "cycle (ms)": round(cycle, 1), "fresh enough": cycle <= NEED_MS})
print(pd.DataFrame(rows).to_string(index=False))
RESULTS["Modbus cycle at 19200 bit/s (ms)"] = rows[1]["cycle (ms)"]
RESULTS["Modbus cycle at 9600 bit/s (ms)"] = rows[0]["cycle (ms)"]

## 4. The remaining useful life of the battery

*Lecture: [Week 13, Predictive maintenance and the remaining useful life](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#remaining-useful-life) and [Week 13, The uncertainty of the remaining useful life](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#rul-uncertainty)*

The twin of the battery also follows its ageing. A capacity test every six months measures the usable energy. A
straight line through the tests extrapolates the month in which the capacity reaches the end of life, 80 percent of the
first capacity. The fit is uncertain, so the cell draws many lines from the covariance of its coefficients and collects
their months of the end of life. Their median is the expected end, and their 5 percent quantile is an end that comes
sooner with a probability of only 5 percent. New modules must be ordered one delivery time before that quantile. The
equivalent full cycles per day of Section 2 tell whether the new schedule ages the battery faster than the schedule of
the tests.

In [ ]:
# TODO: the capacity tests of your battery (month, usable energy in kWh)
TESTS = [(0, 200.0), (6, 198.1), (12, 195.6), (18, 193.9), (24, 191.2), (30, 189.4)]
END = 0.8 * TESTS[0][1]      # end of life (kWh)
LEAD_MONTHS = 6              # delivery time of new modules (months)
CYCLES_OF_TESTS = 0.6        # equivalent full cycles per day during the months of the tests

m, cap = np.array(TESTS, dtype=float).T
coef, cov = np.polyfit(m, cap, 1, cov=True)
draws = np.random.default_rng(14).multivariate_normal(coef, cov, 5000)
ends = (END - draws[:, 1]) / draws[:, 0]                       # months until the line reaches END
ends = ends[draws[:, 0] < 0]
p5, p50, p95 = np.percentile(ends, [5, 50, 95])
print(f"Fade {-coef[0]:.3f} kWh per month; end of life: median month {p50:.0f}, 90 percent between {p5:.0f} and {p95:.0f}")
print(f"Order new modules by month {p5 - LEAD_MONTHS:.0f}")
print(f"Cycles per day: {week['cycles'].mean():.2f} with the new schedule, {CYCLES_OF_TESTS} during the tests")
RESULTS["fade (kWh per month)"] = round(-coef[0], 3)
RESULTS["end of life: median month"] = round(p50, 1)
RESULTS["end of life: 5 percent quantile (month)"] = round(p5, 1)
RESULTS["latest order (month)"] = round(p5 - LEAD_MONTHS, 1)

mm = np.linspace(0, p95, 100)
fig, ax = plt.subplots(figsize=(8, 3.2))
for a, b in draws[:40]:
    ax.plot(mm, a * mm + b, color="#C9D1D8", lw=0.6)
ax.plot(m, cap, "o", color="#1E6FA8", label="capacity tests")
ax.axhline(END, color="#B23B3B", ls="--", lw=1, label="end of life")
ax.axvline(p5, color="#C98510", lw=1, label="5 percent quantile")
ax.set_xlabel("Month")
ax.set_ylabel("Usable energy (kWh)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. The safety argument: A monitor for the reserve

*Lecture: [Week 14, Runtime monitoring and the simplex architecture](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#runtime-monitoring) and [Week 14, Dependability: Faults, errors and failures](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#dependability)*

The cold rooms need the reserve $e_{min}$ in the battery at all times, so that they survive a grid outage. The linear
program respects the reserve only in its model. The real battery has a lower efficiency than the model, so a plan that
ends at the reserve takes the real battery below it. The simplex architecture keeps the optimiser but checks each of its
commands with a simple monitor. The monitor limits a discharge so that the energy stays above the reserve:

$$d_t \le \left(e_{t-1} - e_{min}\right)\eta .$$

Three tests check the monitor before it runs, in the way of the Python step of Week 14. The cell then runs the plans of
Section 2 on the real battery, with and without the monitor. Each day, the planner starts from the measured energy at
midnight.

In [ ]:
# TODO: the real battery and the safety requirement
ETA_REAL = 0.92              # efficiency of the real battery, lower than in the model
E_RES = E_MIN                # reserve that must remain for the cold rooms (kWh)

def allowed(e, c, d, eta=ETA_REAL):
    """The monitor: limits a planned charge and discharge so that the energy stays between E_RES and E_CAP."""
    d_ok = min(d, max(0.0, (e - E_RES) * eta))
    c_ok = min(c, max(0.0, (E_CAP - e) / eta))
    return c_ok, d_ok

def test_keeps_reserve():
    for e in [E_RES, E_RES + 5.0, 150.0, E_CAP]:
        _, d = allowed(e, 0.0, P_BAT)
        assert e - d / ETA_REAL >= E_RES - 1e-9

def test_safe_command_unchanged():
    assert allowed(150.0, 0.0, 20.0) == (0.0, 20.0)

def test_never_adds_power():
    for e in np.linspace(0, E_CAP, 21):
        c, d = allowed(e, 30.0, 30.0)
        assert 0 <= c <= 30.0 and 0 <= d <= 30.0

for test in [test_keeps_reserve, test_safe_command_unchanged, test_never_adds_power]:
    test()
print("All tests of the monitor passed.")

def run_week(monitor):
    e, energy, changed, cost = E0, [], 0, 0.0
    for k in range(7):
        a = TEST.iloc[24 * k: 24 * (k + 1)]
        plan = schedule(LOAD_FC[24 * k: 24 * (k + 1)], PV_FC[24 * k: 24 * (k + 1)], e0=min(max(e, E_MIN), E_CAP))
        cs, ds = [], []
        for c, d in zip(plan["c"], plan["d"]):
            if monitor:
                c2, d2 = allowed(e, c, d)
                changed += abs(c2 - c) + abs(d2 - d) > 1e-6
                c, d = c2, d2
            e = e + ETA_REAL * c - d / ETA_REAL
            energy.append(e)
            cs.append(c)
            ds.append(d)
        cost += day_cost(a.load.values, a.pv.values, np.array(cs), np.array(ds))
    energy = np.array(energy)
    return {"lowest energy (kWh)": round(float(energy.min()), 1), "hours below the reserve": int((energy < E_RES - 1e-6).sum()),
            "commands changed": changed, "cost of the week (euro)": round(cost, 2)}, energy

res_without, e_without = run_week(False)
res_with, e_with = run_week(True)
print(pd.DataFrame([{"monitor": "off", **res_without}, {"monitor": "on", **res_with}]).to_string(index=False))
RESULTS["without the monitor: hours below the reserve"] = res_without["hours below the reserve"]
RESULTS["with the monitor: hours below the reserve"] = res_with["hours below the reserve"]
RESULTS["with the monitor: commands changed"] = res_with["commands changed"]
RESULTS["cost of the monitor (euro per week)"] = round(res_with["cost of the week (euro)"] - res_without["cost of the week (euro)"], 2)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(np.arange(168) / 24, e_without, label="without the monitor")
ax.plot(np.arange(168) / 24, e_with, label="with the monitor")
ax.axhline(E_RES, color="#B23B3B", ls="--", lw=1, label="reserve")
ax.set_xlabel("Day of the last week")
ax.set_ylabel("Real energy (kWh)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Write the safety argument of the report as a claim, supported by evidence. The evidence here is the tests of the monitor, the run with the real battery and the assumptions behind them. Name the assumption that you trust least, for example the measured energy at midnight.

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>